# Data Preprocessing

*Generated by DoML — Do Machine Learning*

> **Note:** This notebook uses Python regardless of the `analysis_language` config setting.
> scikit-learn, SHAP, and Optuna do not have comparable R equivalents.

In [ ]:
# Reproducibility — REPR-01 (non-negotiable per CLAUDE.md)
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import json
import glob
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import Markdown, display

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')

In [ ]:
# Project root — REPR-02 (non-negotiable per CLAUDE.md)
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/home/jovyan/work'))

with open(PROJECT_ROOT / '.planning' / 'config.json') as f:
    config = json.load(f)

problem_type = config.get('problem_type', 'regression')
dataset_path = config.get('dataset', {}).get('path', '')
print(f"Project root: {PROJECT_ROOT}")
print(f"Problem type: {problem_type}")
print(f"Dataset path: {dataset_path}")

In [ ]:
# Load the cleaned dataset produced by Phase 5 (EDA)
# data/processed/ is writable — this is the correct input source
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_files = list(processed_dir.glob('*.csv')) + list(processed_dir.glob('*.parquet'))

if not processed_files:
    raise FileNotFoundError(
        f"No files found in {processed_dir}. Run /doml-execute-phase 2 first."
    )

# Use the first non-preprocessed file found (exclude already-preprocessed outputs)
input_file = next(
    (f for f in processed_files if not f.name.startswith('preprocessed_')),
    processed_files[0]
)
print(f"Loading: {input_file.name}")

if input_file.suffix == '.parquet':
    df = pd.read_parquet(input_file)
else:
    df = pd.read_csv(input_file)

print(f"Shape: {df.shape[0]:,} rows \u00d7 {df.shape[1]} columns")
display(df.head(3))

## Section 1: Imputation

Address missing values before encoding and scaling. Numeric columns have three candidate strategies compared below. Categorical columns use most-frequent imputation (standard; no comparison needed).

**Analyst action:** Review the comparison plots and document your chosen strategy per column in the cell below.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)

numeric_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_missing = [c for c in numeric_cols if c in missing_df.index]
categorical_missing = [c for c in categorical_cols if c in missing_df.index]

print("Columns with missing values:")
display(missing_df)
print(f"\nNumeric with missing: {numeric_missing}")
print(f"Categorical with missing: {categorical_missing}")

In [ ]:
# Compare imputation strategies for numeric columns with significant missingness
cols_to_compare = [c for c in numeric_missing if missing_pct[c] > 5]

if cols_to_compare:
    for col in cols_to_compare[:3]:  # limit to 3 columns for readability
        imp_median = SimpleImputer(strategy='median')
        imp_mean = SimpleImputer(strategy='mean')
        imp_knn = KNNImputer(n_neighbors=5)
        
        df_col = df[[col]]
        filled_median = imp_median.fit_transform(df_col).flatten()
        filled_mean = imp_mean.fit_transform(df_col).flatten()
        filled_knn = imp_knn.fit_transform(df_col).flatten()
        
        fig, axes = plt.subplots(1, 3, figsize=(12, 3))
        for ax, vals, label in zip(axes,
                                    [filled_median, filled_mean, filled_knn],
                                    ['Median', 'Mean', 'KNN(k=5)']):
            ax.hist(vals, bins=30, color='steelblue', alpha=0.7)
            ax.set_title(f'{col}: {label}')
            ax.set_xlabel(col)
        plt.suptitle(f'Imputation Comparison \u2014 {col}', y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric columns with >5% missing \u2014 skipping comparison plots.")
    print("SimpleImputer(strategy='median') will be used as default.")

In [ ]:
if categorical_missing:
    print("Categorical columns missing summary:")
    for col in categorical_missing:
        top_val = df[col].mode()[0] if not df[col].mode().empty else 'N/A'
        print(f"  {col}: {missing_pct[col]:.1f}% missing \u2014 will impute with most_frequent ('{top_val}')")
else:
    print("No categorical columns with missing values.")

### Analyst Decision: Imputation Strategy

Document your chosen imputation strategy for each column with missing values:

| Column | Type | % Missing | Chosen Strategy | Rationale |
|--------|------|-----------|-----------------|----------|
| *column_name* | numeric | *N%* | median / mean / KNN | *reason* |
| *column_name* | categorical | *N%* | most_frequent | standard |

**Notes:**
- Median is robust to outliers (preferred when EDA flagged skewed distributions)
- Mean is appropriate for approximately normal distributions
- KNN is best when missingness may be related to other features (MAR pattern)

## Section 2: Encoding Categorical Variables

**Rules (DoML defaults):**
- **Low cardinality (\u2264 10 unique values):** OneHotEncoder \u2014 creates binary indicator columns
- **High cardinality (> 10 unique values):** OrdinalEncoder \u2014 assigns integer rank; note this implies ordinality which may not exist. For supervised problems, TargetEncoder is a stronger alternative.

**Analyst action:** Review the cardinality summary and document encoding choices below.

In [ ]:
if categorical_cols:
    cardinality = df[categorical_cols].nunique().sort_values()
    cardinality_df = pd.DataFrame({
        'unique_values': cardinality,
        'strategy': ['OneHotEncoder' if n <= 10 else 'OrdinalEncoder (high cardinality)'
                     for n in cardinality]
    })
    display(cardinality_df)
    
    low_card = [c for c in categorical_cols if df[c].nunique() <= 10]
    high_card = [c for c in categorical_cols if df[c].nunique() > 10]
    print(f"\nLow cardinality (OneHotEncoder): {low_card}")
    print(f"High cardinality (OrdinalEncoder): {high_card}")
else:
    print("No categorical columns detected.")
    low_card, high_card = [], []

### Analyst Decision: Encoding Strategy

| Column | Unique Values | Chosen Encoder | Notes |
|--------|---------------|----------------|-------|
| *column_name* | *N* | OneHotEncoder / OrdinalEncoder | *e.g., natural order exists* |

## Section 3: Scaling / Normalization

**Important:** Tree-based models (RandomForest, XGBoost, LightGBM) do **not** require feature scaling \u2014 they are invariant to monotonic transformations. Scaling is only necessary for linear models (Ridge, Lasso, Logistic Regression) and distance-based models (KNN, SVM).

Two scalers are compared below:
- **StandardScaler**: zero mean, unit variance. Sensitive to outliers.
- **RobustScaler**: uses median and IQR. Preferred when EDA flagged outliers.

In [ ]:
numeric_all = df.select_dtypes(include='number').columns.tolist()
# Exclude target if present
TARGET = config.get('target_column', None)
numeric_features = [c for c in numeric_all if c != TARGET]

if numeric_features and len(numeric_features) <= 10:
    sample_df = df[numeric_features].dropna()
    
    ss = StandardScaler()
    rs = RobustScaler()
    
    df_std = pd.DataFrame(ss.fit_transform(sample_df), columns=numeric_features)
    df_rob = pd.DataFrame(rs.fit_transform(sample_df), columns=numeric_features)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    df_std.boxplot(ax=axes[0])
    axes[0].set_title('StandardScaler')
    axes[0].tick_params(axis='x', rotation=45)
    df_rob.boxplot(ax=axes[1])
    axes[1].set_title('RobustScaler')
    axes[1].tick_params(axis='x', rotation=45)
    plt.suptitle('Feature Distributions After Scaling')
    plt.tight_layout()
    plt.show()
elif numeric_features:
    print(f"{len(numeric_features)} numeric features \u2014 displaying summary statistics instead of boxplots.")
    print("\nStandardScaler result summary:")
    display(pd.DataFrame(StandardScaler().fit_transform(df[numeric_features].dropna()),
                         columns=numeric_features).describe().round(3))
else:
    print("No numeric features to scale.")

## Section 4: Feature Engineering & Selection

This section helps identify:
1. **Interaction candidates** \u2014 top correlated feature pairs from EDA (potential multiplicative features)
2. **Feature importance** \u2014 RandomForest or mutual information ranking
3. **Redundant features** \u2014 pairs with |r| > 0.95 (remove one of the pair)
4. **Multicollinearity (VIF)** \u2014 for regression problems only; VIF > 10 signals a problem

In [ ]:
numeric_df = df[numeric_features].dropna() if numeric_features else pd.DataFrame()

if len(numeric_features) >= 2:
    corr_matrix = numeric_df.corr().abs()
    # Get upper triangle
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    top_pairs = upper.stack().sort_values(ascending=False).head(5)
    
    print("Top 5 correlated feature pairs (interaction candidates):")
    for (f1, f2), corr_val in top_pairs.items():
        print(f"  {f1} \u00d7 {f2}: r = {corr_val:.3f}")
    
    # Flag highly correlated pairs (|r| > 0.95)
    redundant = [(f1, f2, v) for (f1, f2), v in upper.stack().items() if v > 0.95]
    if redundant:
        print("\n\u26a0\ufe0f  Highly correlated pairs (|r| > 0.95) \u2014 consider dropping one:")
        for f1, f2, v in redundant:
            print(f"  {f1} / {f2}: r = {v:.3f}")
    else:
        print("\nNo redundant feature pairs (|r| > 0.95) found.")
else:
    print("Fewer than 2 numeric features \u2014 skipping correlation analysis.")

In [ ]:
if TARGET and TARGET in df.columns and numeric_features:
    X = df[numeric_features].dropna()
    y = df.loc[X.index, TARGET]
    
    if problem_type in ('regression',):
        importance_scores = mutual_info_regression(X, y, random_state=SEED)
    else:
        importance_scores = mutual_info_classif(X, y, random_state=SEED)
    
    importance_df = pd.DataFrame({
        'feature': numeric_features,
        'importance': importance_scores
    }).sort_values('importance', ascending=False)
    
    fig, ax = plt.subplots(figsize=(8, max(4, len(numeric_features) * 0.3)))
    ax.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
    ax.set_xlabel('Mutual Information Score')
    ax.set_title('Feature Importance (Mutual Information)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    display(importance_df)
else:
    print("TARGET not set in config.json \u2014 skipping importance ranking.")
    print("Set config['target_column'] in .planning/config.json to enable this section.")

In [ ]:
if problem_type == 'regression' and len(numeric_features) >= 2:
    X_vif = df[numeric_features].dropna()
    vif_data = pd.DataFrame({
        'feature': numeric_features,
        'VIF': [variance_inflation_factor(X_vif.values, i)
                for i in range(len(numeric_features))]
    }).sort_values('VIF', ascending=False)
    
    print("Variance Inflation Factor (VIF) \u2014 flags multicollinearity:")
    print("  VIF > 10: high multicollinearity (consider dropping or combining)")
    print("  VIF 5\u201310: moderate (monitor)")
    print("  VIF < 5:  acceptable")
    display(vif_data.round(2))
    
    high_vif = vif_data[vif_data['VIF'] > 10]
    if not high_vif.empty:
        print(f"\n\u26a0\ufe0f  {len(high_vif)} feature(s) with VIF > 10: {high_vif['feature'].tolist()}")
    else:
        print("\nAll features have VIF \u2264 10 \u2014 no severe multicollinearity detected.")
elif problem_type != 'regression':
    print(f"VIF analysis skipped \u2014 not applicable for {problem_type} problems.")
else:
    print("Fewer than 2 numeric features \u2014 VIF not computed.")

### Analyst Decision: Feature Selection

Document features to keep, drop, or engineer:

| Action | Feature(s) | Reason |
|--------|-----------|--------|
| Drop | *feature_name* | *high VIF / redundant with X / low importance* |
| Keep | *feature_name* | *important predictor* |
| Engineer | *feat_a \u00d7 feat_b* | *high correlation \u2014 candidate interaction* |

**Features to drop** (update `features_to_drop` in next cell):

In [ ]:
# \u2500\u2500 ANALYST CONFIGURATION \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Update these lists based on your decisions above

features_to_drop = []           # e.g. ['redundant_feature', 'low_importance_col']
impute_numeric_strategy = 'median'  # 'median', 'mean', or 'knn'
scaler_choice = 'robust'        # 'standard' or 'robust'

# \u2500\u2500 PIPELINE ASSEMBLY \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
df_model = df.drop(columns=features_to_drop, errors='ignore')

final_numeric = [c for c in df_model.select_dtypes(include='number').columns
                 if c != TARGET]
final_categorical = df_model.select_dtypes(include=['object', 'category']).columns.tolist()
final_low_card = [c for c in final_categorical if df_model[c].nunique() <= 10]
final_high_card = [c for c in final_categorical if df_model[c].nunique() > 10]

# Numeric pipeline
if impute_numeric_strategy == 'knn':
    num_imputer = KNNImputer(n_neighbors=5)
else:
    num_imputer = SimpleImputer(strategy=impute_numeric_strategy)

if scaler_choice == 'robust':
    scaler = RobustScaler()
else:
    scaler = StandardScaler()

numeric_pipeline = Pipeline([('imputer', num_imputer), ('scaler', scaler)])

# Categorical pipeline
cat_transformers = []
if final_low_card:
    cat_transformers.append(
        ('onehot', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), final_low_card)
    )
if final_high_card:
    cat_transformers.append(
        ('ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), final_high_card)
    )

# Assemble ColumnTransformer
transformers = [('numeric', numeric_pipeline, final_numeric)] + cat_transformers
preprocessor = ColumnTransformer(transformers=transformers, remainder='passthrough')

print("ColumnTransformer assembled:")
print(f"  Numeric features ({len(final_numeric)}): {scaler_choice} scaler + {impute_numeric_strategy} imputer")
print(f"  Low-cardinality categorical ({len(final_low_card)}): OneHotEncoder")
print(f"  High-cardinality categorical ({len(final_high_card)}): OrdinalEncoder")
print(f"  Dropped features: {features_to_drop}")

In [ ]:
# Fit and transform (on full dataset \u2014 train/test split happens in modelling notebook)
if TARGET and TARGET in df_model.columns:
    X = df_model.drop(columns=[TARGET])
    y = df_model[TARGET]
else:
    X = df_model
    y = None

X_preprocessed = preprocessor.fit_transform(X)

# Get output column names
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = [f'feature_{i}' for i in range(X_preprocessed.shape[1])]

df_preprocessed = pd.DataFrame(X_preprocessed, columns=feature_names)
if y is not None:
    df_preprocessed[TARGET] = y.values

# Write to data/processed/preprocessed_{original_filename} \u2014 PREP-02
output_path = processed_dir / f'preprocessed_{input_file.name}'
if input_file.suffix == '.parquet':
    output_path = processed_dir / f'preprocessed_{input_file.stem}.parquet'
    df_preprocessed.to_parquet(output_path, index=False)
else:
    output_path = processed_dir / f'preprocessed_{input_file.stem}.csv'
    df_preprocessed.to_csv(output_path, index=False)

print(f"Preprocessed dataset written to: {output_path.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df_preprocessed.shape[0]:,} rows \u00d7 {df_preprocessed.shape[1]} columns")
display(df_preprocessed.head(3))

In [ ]:
print("=" * 60)
print("PREPROCESSING SUMMARY")
print("=" * 60)
print(f"Input:  {input_file.name}  ({df.shape[0]:,} \u00d7 {df.shape[1]})")
print(f"Output: {output_path.name}  ({df_preprocessed.shape[0]:,} \u00d7 {df_preprocessed.shape[1]})")
print(f"Dropped features: {features_to_drop if features_to_drop else 'none'}")
print(f"Imputation (numeric): {impute_numeric_strategy}")
print(f"Scaling: {scaler_choice}")
print(f"Encoding \u2014 OneHotEncoder: {final_low_card}")
print(f"Encoding \u2014 OrdinalEncoder: {final_high_card}")
print(f"\nNext step: Run /doml-execute-phase 3 to run modelling notebook.")

---

## Caveats & Limitations

- **Correlation is not causation.** Feature importance scores and correlation values indicate statistical association only \u2014 they do not imply that one variable causes another.
- Imputation introduces synthetic values; the impact grows with missingness percentage. Review the before/after distributions above carefully.
- OrdinalEncoder for high-cardinality variables assumes an implicit ordering. If no natural order exists, this encoding can mislead models.
- Scaling parameters (mean, std, median, IQR) are fitted on the **full dataset** at this stage. In the modelling notebook, the pipeline will be re-fitted on training data only to prevent data leakage.
- Feature importance via mutual information is univariate \u2014 it does not capture interaction effects between features.